# Le Shoppe
Le Shoppe is a totally-not-fictional* online retail store. They are interested in finding out why customers might not be returning to buy from their store (aka 'churning'), and would ideally like to identify which customers are "at-risk" of churning so they can pre-emptively entice them to not do that.

<sup>* Any similarity to any actual real-world business for this totally-not-fictional online store is purely coincidental.</sup>

## Data sources

- [Multi source Customer Mart for Female Recommendations in Marketplace](https://data.mendeley.com/datasets/krt74v58kt/2) - MultiSource_Female_CustMart.csv
- [Ecommerce Customer Behavior Dataset](https://www.kaggle.com/datasets/dhairyajeetsingh/ecommerce-customer-behavior-dataset) - ecommerce_customer_churn_dataset.csv


- [eCommerce Customer Service Satisfaction](https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction) - Customer_support_data.csv
- [E-Commerce Shipping Data](https://www.kaggle.com/datasets/prachi13/customer-analytics) - shipping_data.csv
    - Particular fields of interest could be Customer_rating, Cost_of_the_Product, Prior_purchases and Customer_care_calls

- [Amazon E-commerce Products & Reviews Dataset](https://www.kaggle.com/datasets/lazylad99/amazon-e-commerce-product-and-review-dataset) - products.csv, reviews.csv

In [1]:
import pandas as pd

ecommerce_churn = pd.read_csv('datasets/ecommerce_customer_churn_dataset.csv')
customer_behaviour = pd.read_csv('datasets/shopping_trends.csv')
customer_support = pd.read_csv('datasets/Customer_support_data.csv')
shipping_data = pd.read_csv('datasets/shipping_data.csv')

products = pd.read_csv('datasets/products.csv')
reviews = pd.read_csv('datasets/reviews.csv')

### eCommerce Customer Service Satisfaction

In [3]:
customer_support.columns

Index(['Unique id', 'channel_name', 'category', 'Sub-category',
       'Customer Remarks', 'Order_id', 'order_date_time', 'Issue_reported at',
       'issue_responded', 'Survey_response_Date', 'Customer_City',
       'Product_category', 'Item_price', 'connected_handling_time',
       'Agent_name', 'Supervisor', 'Manager', 'Tenure Bucket', 'Agent Shift',
       'CSAT Score'],
      dtype='object')

In [4]:
customer_support['category'].unique()

array(['Product Queries', 'Order Related', 'Returns', 'Cancellation',
       'Shopzilla Related', 'Payments related', 'Refund Related',
       'Feedback', 'Offers & Cashback', 'Onboarding related', 'Others',
       'App/website'], dtype=object)

In [5]:
customer_support['Sub-category'].unique()

array(['Life Insurance', 'Product Specific Information',
       'Installation/demo', 'Reverse Pickup Enquiry', 'Not Needed',
       'Fraudulent User', 'Exchange / Replacement', 'Missing',
       'General Enquiry', 'Return request', 'Delayed',
       'Service Centres Related', 'Payment related Queries',
       'Order status enquiry', 'Return cancellation', 'Unable to track',
       'Seller Cancelled Order', 'Wrong', 'Invoice request',
       'Priority delivery', 'Refund Related Issues', 'Signup Issues',
       'Online Payment Issues', 'Technician Visit',
       'UnProfessional Behaviour', 'Damaged', 'Product related Issues',
       'Refund Enquiry', 'Customer Requested Modifications',
       'Instant discount', 'Card/EMI', 'Shopzila Premium Related',
       'Account updation', 'COD Refund Details', 'Seller onboarding',
       'Order Verification', 'Other Cashback', 'Call disconnected',
       'Wallet related', 'PayLater related', 'Call back request',
       'Other Account Related Issues

In [6]:
customer_support['Product_category'].unique()

array([nan, 'LifeStyle', 'Electronics', 'Mobile', 'Home Appliences',
       'Furniture', 'Home', 'Books & General merchandise', 'GiftCard',
       'Affiliates'], dtype=object)

In [7]:
"""Tidy up this dataset so it's more manageable/relevant to our needs"""

print(f"Customer support records before cleaning: {customer_support.shape[0]}")

# Our pretend eCommerce store doesn't sell life insurance, do home installs, have different marketplace sellers, 
# or offer affiliate deals, so these entries can go
drop_values = ['Life Insurance','Installation/demo','Technician Visit','Service Centres Related', 'Not Needed',
               'Seller Cancelled Order','Customer Requested Modifications','Seller onboarding','Affiliate Offers',
               'Service Center - Service Denial','Policy Related','Self-Help', 'Commission related', 'Card/EMI',
               'Other Cashback']
# Keep rows where these 'Sub-category' values are NOT in the list
customer_support = customer_support[~customer_support['Sub-category'].isin(drop_values)]

# We can get rid of 'Affiliates' out of Product_category, too
customer_support = customer_support[~customer_support['Product_category'].isin(['Affiliates'])]

# The mispelled word is annoying me
customer_support['Product_category'] = customer_support['Product_category'].replace('Home Appliences', 'Home Appliances')

# And then for the sake of consistency, replace 'Shopzilla' with 'Le Shoppe'
customer_support.replace('Shopzilla', 'Le Shoppe', regex=True, inplace=True)
customer_support.replace('Shopzila', 'Le Shoppe', regex=True, inplace=True)

print(f"Customer support records after the purge: {customer_support.shape[0]}")

Customer support records before cleaning: 85907
Customer support records after the purge: 74974


### Shipping Data

In [8]:
shipping_data.columns

Index(['ID', 'Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls',
       'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases',
       'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms',
       'Reached.on.Time_Y.N'],
      dtype='object')

In [9]:
print(shipping_data.iloc[0])

ID                          1
Warehouse_block             D
Mode_of_Shipment       Flight
Customer_care_calls         4
Customer_rating             2
Cost_of_the_Product       177
Prior_purchases             3
Product_importance        low
Gender                      F
Discount_offered           44
Weight_in_gms            1233
Reached.on.Time_Y.N         1
Name: 0, dtype: object


In [10]:
shipping_data['Customer_care_calls'].unique()

array([4, 2, 3, 5, 6, 7])

### Product Data

In [11]:
products.columns

Index(['s.no', 'about_item', 'asin', 'availability', 'best_sellers_rank',
       'brand_name', 'brand_page_url', 'breadcrumbs',
       'customer_review_summary', 'default_variant/0', 'default_variant/1',
       'default_variant/2', 'delivery_date', 'fastest_delivery_date',
       'list_price', 'manufacturer', 'model_number', 'price_value',
       'product_description', 'product_url', 'rating_count',
       'rating_distribution/1star', 'rating_distribution/2star',
       'rating_distribution/3star', 'rating_distribution/4star',
       'rating_distribution/5star', 'rating_stars', 'recent_purchases',
       'scrape_time', 'seller_name', 'seller_page_url', 'title', 'all_images',
       'rank_1'],
      dtype='object')

In [ ]:
products_feats = ['asin','title','about_item','brand_name','list_price','availability']
products_subset = products[products_feats].copy()

products_subset['list_price'] = products_subset['list_price'].str.replace(r'[^0-9.]', '', regex=True) # Keep only the float
products_subset['list_price'] = products_subset['list_price'].astype('float64')

In [ ]:
is_accessories = products_subset['title'].str.contains('Belt', case=False, na=False)
products_subset.loc[is_accessories, 'category'] = 'Accessories'

is_footwear = products_subset['title'].str.contains('Sneaker|Shoe|Sandal|Skechers|Trainer|Loafer|Slip-on|Oxford|C 85|Brecon', case=False, na=False)
products_subset.loc[is_footwear, 'category'] = 'Footwear'
products_subset.loc[is_footwear, 'sub-category'] = 'Footwear'

is_outerwear = products_subset['title'].str.contains('Jacket|Coat|Puffer', case=False, na=False)
products_subset.loc[is_outerwear, 'category'] = 'Outerwear'
products_subset.loc[is_outerwear, 'sub-category'] = 'Jackets & Coats'

is_clothing = products_subset['title'].str.contains('Pant|Shorts|Sleep short|cargo short|Shirt|Tee|Jean|Jegging|Polo|Dress|Blouse|Hoodie|Sock|Underwear|Brief|Boxer|Joggers', case=False, na=False)
products_subset.loc[is_clothing, 'category'] = 'Clothing'
is_socks = products_subset['title'].str.contains('Shirt|Tee|Polo|Blouse|Hoodie', case=False, na=False)
products_subset.loc[is_socks, 'sub-category'] = 'Tops'
is_socks = products_subset['title'].str.contains('Pants|Shorts|cargo short|Jean|Jegging|Joggers', case=False, na=False)
products_subset.loc[is_socks, 'sub-category'] = 'Bottoms'
is_underwear = products_subset['title'].str.contains('Underwear|Brief|Boxer', case=False, na=False)
products_subset.loc[is_underwear, 'sub-category'] = 'Underwear'
is_socks = products_subset['title'].str.contains('Socks', case=False, na=False)
products_subset.loc[is_socks, 'sub-category'] = 'Socks'

products_subset['category'] = products_subset['category'].fillna('Clothing')
products_subset['sub-category'] = products_subset['sub-category'].fillna('Other')

In [14]:
products_no_nans = products_subset.dropna()

In [15]:
# Fill at least some NaNs
group_means = products_no_nans.groupby(['brand_name', 'sub-category'])['list_price'].mean()
fill_values = products_subset.set_index(['brand_name', 'sub-category']).index.map(group_means)
products_subset['list_price'] = products_subset['list_price'].fillna(pd.Series(fill_values, index=products_subset.index))
products_subset.isna().sum()

asin              0
title             0
about_item        0
brand_name        0
list_price      196
availability     13
category          0
sub-category      0
dtype: int64

In [16]:
products_subset.dropna(inplace=True)

In [17]:
products_subset['brand_name'].unique()

array(['MLYENX Store', 'COOFANDY Store', 'ZITY Store', 'V VALANCH Store',
       'POLO Store', 'POLO RALPH LAUREN Store', 'TELALEO Store',
       'OYGSieg Store', 'Hanes Store', 'U.S. Polo Assn. Store',
       'GEEK LIGHTING Store', 'HUAKANG Store', 'Brand: U.S. Polo Assn.',
       'Gildan Store', 'Nautica Store', 'Under Armour Store',
       'Lacoste Store', 'ROOMLINE Store', 'MAGCOMSEN Store',
       'Brand: Nike', 'BALENNZ Store', 'Qekoma Store', 'TACVASEN Store',
       'Fruit of the Loom Store', 'Liberty Imports Store',
       'Real Essentials Store', 'True Classic Store', 'Champion Store',
       'Brand: MCPORO', 'HovSiyla Store', 'Carhartt Store',
       'Amazon Essentials Store', 'Kingsted Store', 'Dickies Store',
       'Brand: Kinglaman', 'duwqcui Store', 'Alimens & Gentle Store',
       'Brand: Olidarua', 'Levi Strauss Signature Gold Store',
       'Rustler Store', 'Wrangler Store', "Levi's Store", 'Lee Store',
       'Rock & Republic Store', 'Brand: OIIIO', 'Lucky Brand Sto

In [18]:
products_subset.groupby('category')['title'].count()

category
Clothing     433
Footwear      66
Outerwear     25
Name: title, dtype: int64

In [19]:
products_subset.isna().sum()

asin            0
title           0
about_item      0
brand_name      0
list_price      0
availability    0
category        0
sub-category    0
dtype: int64

In [20]:
products_df = products_subset[['asin', 'category', 'brand_name', 'title', 'about_item', 'list_price']].copy()
products_df['list_price'] = products_df['list_price'].astype('float64')

### Product Reviews

In [21]:
reviews.columns

Index(['s.no', 'helpfulVoteCount', 'images/0', 'images/1', 'images/2',
       'images/3', 'images/4', 'images/5', 'images/6', 'images/7',
       'productASIN', 'productVariant', 'rating', 'reviewID', 'reviewMetadata',
       'reviewPosition', 'reviewText', 'reviewTitle', 'reviewURL',
       'verifiedPurchase', 'videos/0', 'cleaned_review_text',
       'sentiment_score'],
      dtype='object')

In [22]:
reviews['reviewMetadata'].iloc[0]

'Reviewed in the United States on March 6, 2025'

In [23]:
reviews_feats = ['reviewID','productASIN','rating','reviewTitle','reviewText','reviewMetadata']
reviews_subset = reviews[reviews_feats].copy()

# Keep only the reviews we still have product IDs for
asin_array = products_subset['asin'].to_numpy()
reviews_subset = reviews_subset[reviews_subset['productASIN'].isin(asin_array)].copy() 

# Extract the date from the metadata
reviews_subset['date_string'] = reviews_subset['reviewMetadata'].str.extract(r"on ([A-Za-z]+ \d+, \d{4})")
reviews_subset['date'] = pd.to_datetime(reviews_subset['date_string'], format='%B %d, %Y')
reviews_subset['date_str'] = reviews_subset['date'].dt.strftime('%Y-%m-%d') # change the date formatting to year, month, day
reviews_subset['date'] = pd.to_datetime(reviews_subset['date_str'])

# Final clean up
reviews_subset.dropna(inplace=True)
reviews_df = reviews_subset[['reviewID','date','productASIN','rating','reviewTitle','reviewText']].copy()

reviews_df.head()


,reviewID,date,productASIN,rating,reviewTitle,reviewText
0,R2AUQFPJY5ERCZ,2025-03-06,B0DLGB4RYH,5.0,Stylish and Lightweight COOFANDY Polo Shirt,‚úçÔ∏è The COOFANDY Men's Polo Shirt is a fant...
1,R1YNLJ0QFLB1VT,2025-03-07,B0DRXF62JH,5.0,New favorite shirts,I love these. They look good. They fit well. T...
2,R11LIYW9RMDK9R,2025-02-27,B0DRXF62JH,4.0,Incredibly Good Deal,My expectations were low -- how good could shi...
3,R3K9X391DJNTPZ,2025-02-26,B0DRXF62JH,5.0,Higher quality & breathable 3-pack shirt set t...,This 3-pack of multi-color shirts is really a ...
4,RP17IW9ZKENDZ,2025-02-20,B0DRXF62JH,5.0,Pocket Polos Looks Good,With these three polo shirts I was very satisf...


### Customer Behaviour

In [24]:
print(customer_behaviour.iloc[0])

Customer ID                           1
Age                                  55
Gender                             Male
Item Purchased                   Blouse
Category                       Clothing
Purchase Amount (USD)                53
Location                       Kentucky
Size                                  L
Color                              Gray
Season                           Winter
Review Rating                       3.1
Subscription Status                 Yes
Payment Method              Credit Card
Shipping Type                   Express
Discount Applied                    Yes
Promo Code Used                     Yes
Previous Purchases                   14
Preferred Payment Method          Venmo
Frequency of Purchases      Fortnightly
Name: 0, dtype: object


In [25]:
customer_behaviour.isna().sum()

Customer ID                 0
Age                         0
Gender                      0
Item Purchased              0
Category                    0
Purchase Amount (USD)       0
Location                    0
Size                        0
Color                       0
Season                      0
Review Rating               0
Subscription Status         0
Payment Method              0
Shipping Type               0
Discount Applied            0
Promo Code Used             0
Previous Purchases          0
Preferred Payment Method    0
Frequency of Purchases      0
dtype: int64

In [26]:
customer_behaviour['Category'].unique()

array(['Clothing', 'Footwear', 'Outerwear', 'Accessories'], dtype=object)

In [27]:
customer_behaviour['Shipping Type'].unique()

array(['Express', 'Free Shipping', 'Next Day Air', 'Standard',
       '2-Day Shipping', 'Store Pickup'], dtype=object)

In [28]:
customer_behaviour['Preferred Payment Method'].unique()

array(['Venmo', 'Cash', 'Credit Card', 'PayPal', 'Bank Transfer',
       'Debit Card'], dtype=object)

In [29]:
# Our pretend eCommerce store only sells clothing, footwear and outerwear
customer_behaviour = customer_behaviour[~customer_behaviour['Category'].isin(['Accessories'])]
# We don't have in-store pickup
customer_behaviour = customer_behaviour[~customer_behaviour['Shipping Type'].isin(['Store Pickup'])]
# So no option to pay with cash, either
customer_behaviour = customer_behaviour[~customer_behaviour['Preferred Payment Method'].isin(['Cash'])]

### All done

In [30]:
print(f"""
Ecommerce churn: {ecommerce_churn.shape[1]} features, {ecommerce_churn.shape[0]} rows
Customer support: {customer_support.shape[1]} features, {customer_support.shape[0]} rows

Customer purchase behaviour: {customer_behaviour.shape[1]} features, {customer_behaviour.shape[0]} rows
Shipping data: {shipping_data.shape[1]} features, {shipping_data.shape[0]} rows
Products: {products_df.shape[1]} features, {products_df.shape[0]} rows
Product reviews: {reviews_df.shape[1]} features, {reviews_df.shape[0]} rows
""")


Ecommerce churn: 25 features, 50000 rows
Customer support: 20 features, 74974 rows

Customer purchase behaviour: 19 features, 1844 rows
Shipping data: 12 features, 10999 rows
Products: 6 features, 524 rows
Product reviews: 6 features, 4660 rows

